In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"

df = pd.read_csv(url)

In [2]:
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Pregunta A

In [3]:
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

#### Respuesta:

La columna Rating tiene 1474 valores nulos, mientras que Type tiene 1 y Android Ver tiene 3.

Es seguro eliminar las filas donde falta Android Ver o Type porque representan una cantidad muy pequeña en comparación con el total de registros del dataset, por lo que la pérdida de información sería mínima.

En cambio, eliminar las filas donde falta Rating sería un error porque se perderían 1474 registros, una cantidad considerable de datos. Esto podría reducir la información disponible para entrenar el modelo e incluso introducir sesgo. Por esta razón, en Rating es más conveniente utilizar una técnica de imputación para rellenar los valores faltantes.

## Pregunta B

In [4]:
df.shape

(10841, 13)

In [9]:
antes = len(df)
print("Registros antes:", antes)

Registros antes: 10841


In [11]:
df = df.dropna(subset=['Android Ver', 'Type'])

In [12]:
despues = len(df)

print("Registros antes:", antes)
print("Registros después:", despues)
print("Registros eliminados:", antes - despues)

Registros antes: 10841
Registros después: 10837
Registros eliminados: 4


In [13]:
df[['Android Ver', 'Type']].isnull().sum()

Android Ver    0
Type           0
dtype: int64

#### Respuesta:

Se utilizó `dropna()` con el parámetro `subset=['Android Ver', 'Type']` para eliminar únicamente las filas que tenían valores nulos en esas dos columnas.

El DataFrame pasó de 10,841 a 10,837 registros, por lo que solamente se eliminaron 4 filas. Esta reducción es mínima en comparación con el tamaño total del dataset, por lo que la pérdida de información es prácticamente insignificante.

Después de la eliminación, las columnas Android Ver y Type tienen 0 valores nulos.

## Pregunta C

In [14]:
df['Rating'].isnull().sum()

np.int64(1473)

In [15]:
df['Rating'] = df['Rating'].fillna(
    df.groupby('Category')['Rating'].transform('median')
)

In [16]:
df['Rating'].isnull().sum()

np.int64(0)

#### Respuesta

Después de eliminar las filas de la Pregunta B, quedaron 1473 valores nulos en la columna Rating.

Para rellenarlos se utilizó la mediana de Rating correspondiente a la categoría de cada aplicación. Esto se hizo agrupando los datos por Category con `groupby()` y utilizando `transform('median')`.

Esta estrategia es más adecuada que utilizar el promedio global porque las aplicaciones de diferentes categorías pueden tener comportamientos de calificación distintos. Por ejemplo, las aplicaciones de FAMILY pueden recibir calificaciones diferentes a las de DATING.

Además, la mediana es menos sensible a valores extremos que el promedio.

Después de aplicar la imputación, la columna Rating quedó con 0 valores nulos.

## Pregunta D

In [17]:
(df['Size'] == 'Varies with device').sum()

np.int64(1694)

In [18]:
df['Size'] = df['Size'].replace('Varies with device', np.nan)

In [19]:
df['Size'].isnull().sum()

np.int64(1694)

#### Respuesta:

La columna Size no tenía valores nulos matemáticos originalmente, pero contenía 1694 registros con el texto "Varies with device".

Se utilizó `replace()` para convertir ese texto en un valor nulo real de NumPy:

`df['Size'] = df['Size'].replace('Varies with device', np.nan)`

Después de realizar la conversión, la columna Size quedó con 1694 valores NaN.

Es mejor utilizar NaN en lugar de asignar un tamaño falso como 0 MB, porque 0 MB sería un dato incorrecto. El modelo podría interpretar que esas aplicaciones realmente tienen un tamaño de cero y generar conclusiones erróneas.

En cambio, NaN representa correctamente que el tamaño es desconocido y permite aplicar posteriormente una estrategia adecuada para tratar esos datos faltantes.